# Post-Project: Test and Evaluation

## Context

**Smart Stethoscope** was a bootcamp group project focused on classifying respiratory conditions from lung sound recordings.

The original project explored multiple modelling paths - including logistic regression, XGBoost, CNNs on mel spectrograms, and a late-fusion hybrid model - and ultimately shipped a hybrid demo application. This notebook revisits that work with three goals:

1. reconstruct the pipeline clearly
2. rerun the models with a consistent grouped evaluation setup
3. understand which parts of the original approach were genuinely adding value

#### Approach

- Recovered the canonical pipeline from the application code and main modelling notebooks

- Reran preprocessing from raw audio to:
  - engineered tabular features
  - mel spectrogram inputs

- Created grouped outer and inner splits using `StratifiedGroupKFold`

- Rebuilt and evaluated:
  - a clean logistic regression baseline
  - XGBoost
  - CNN
  - late-fusion hybrid model

- Compared chunk-level and patient-level performance

- Ran qualitative single-file inference checks on three example recordings

### Pipeline summary

This table summarises the pipeline executed in this notebook, alongside the most relevant original notebooks that informed each step.

| Step | What happens | Relevant notebook(s) | Key decisions in this rerun |
|------|-------------|---------------------|-----------------------------|
| 1. Preprocessing | Raw audio → tabular features + mel spectrograms | `Preprocessing-Lily.ipynb`, `preprocessing-pipeline-Lily.ipynb` | 12 kHz sampling, 6s segments, 2s step, final 5-class setup |
| 2. Splitting | Train / validation / test split | This notebook | `StratifiedGroupKFold` with patient-level grouping to avoid leakage |
| 3. Baseline | Logistic regression on engineered features | `02_keira_baseline_model.ipynb`, `03_keira_baseline_integration_test.ipynb` | Clean baseline aligned to final pipeline (42 engineered audio features) |
| 4. XGBoost | Tree-based tabular model | `06_xgboost_improvements.ipynb`, `10_xgboost_improvements_v2.ipynb` | Class-weighted training, early stopping on validation set |
| 5. CNN | Spectrogram-based model | `04_keira_simple_cnn.ipynb` | 4-block CNN, class weighting added here for stability |
| 6. Hybrid | Late fusion of XGBoost + CNN | `05_keira_xgboost_feat_with_cnn.ipynb`, `06_keira_xgboost_cnn_hybrid_v2.ipynb` | Weighted probability averaging (0.8 XGB / 0.2 CNN) |
| 7. Run summary | Model comparison + inference checks | This notebook, `07_keira_model_test.ipynb`, `08_keira_model_evaluation.ipynb` | Macro F1 (chunk + patient), plus qualitative example predictions |
 

### Executive summary

This notebook reconstructs and re-evaluates the Smart Stethoscope project end to end, using the current Python pipeline as the canonical implementation and a stricter grouped evaluation setup than we used during the original project.

Key choices in this rerun:
- final 5-class setup: Bronchiectasis, COPD, Healthy, Pneumonia, URTI
- 6-second audio chunks
- grouped splitting by patient to avoid leakage
- chunk-level and patient-level evaluation
- macro F1 used as the main comparison metric

#### Model comparison

| Model | Input | Chunk Macro F1 | Patient Macro F1 | Notes |
|------|------|----------------|------------------|------|
| Logistic regression baseline | 42 engineered audio features | 0.51 | 0.45 | Clean benchmark aligned to final pipeline |
| **XGBoost** 🏆 | 42 engineered audio features | **0.41** | **0.49** | Best patient-level performance |
| CNN (class-weighted) | Mel spectrograms | 0.33 | 0.32 | Weaker standalone model |
| Hybrid (late fusion) | XGBoost + CNN probabilities | 0.41 | 0.39 | No improvement over XGBoost |

👉 **Key takeaway**:

  > XGBoost emerged as the strongest standalone model in this re-evaluation. The hybrid approach used in the original project was reconstructed faithfully, but did not improve performance over XGBoost on this setup. The main limitation appears to be the dataset, especially class imbalance and weak separability between conditions, rather than lack of model complexity.

## 0. Project Config and Final Pipeline Settings

This notebook reconstructs the final Smart Stethoscope pipeline using the current Python application code as the canonical implementation.

The settings below are taken from `params.py` and treated as the source of truth for this rerun.

Key final pipeline settings:
- Sampling rate: 12,000 Hz
- Segment length: 6 seconds
- Step length: 2 seconds
- Fusion weighting: 0.8 XGBoost / 0.2 CNN
- Final class set: Bronchiectasis, COPD, Healthy, Pneumonia, URTI

In [2]:
import sys
from pathlib import Path

# Add repo root to Python path
repo_root = Path().resolve().parent
sys.path.append(str(repo_root))

print("Repo root added to path:", repo_root)

Repo root added to path: /Users/keira/Documents/GitHub/smart-stethoscope


In [38]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from xgboost import XGBClassifier

from smart_stethoscope.ml_logic.data_loading import load_and_preprocess_raw_audio_data
from smart_stethoscope.ml_logic.model import train_final_hybrid_models, predict_hybrid

from smart_stethoscope.params import (
    TARGET_SAMPLING_RATE,
    SEGMENT_LENGTH,
    STEP_LENGTH,
    DEFAULT_XGB_WEIGHT,
    CLASS_NAMES,
    DEMO_BLACKLIST,
)

print("TARGET_SAMPLING_RATE:", TARGET_SAMPLING_RATE)
print("SEGMENT_LENGTH:", SEGMENT_LENGTH)
print("STEP_LENGTH:", STEP_LENGTH)
print("DEFAULT_XGB_WEIGHT:", DEFAULT_XGB_WEIGHT)
print("CLASS_NAMES:", CLASS_NAMES)
print("DEMO_BLACKLIST:", DEMO_BLACKLIST)

TARGET_SAMPLING_RATE: 12000
SEGMENT_LENGTH: 6
STEP_LENGTH: 2
DEFAULT_XGB_WEIGHT: 0.8
CLASS_NAMES: ['Bronchiectasis', 'COPD', 'Healthy', 'Pneumonia', 'URTI']
DEMO_BLACKLIST: ['142', '191', '182']


## 1. Load and preprocess data

In [5]:
from smart_stethoscope.params import RAW_AUDIO_PATH, DIAGNOSIS_PATH

print("RAW_AUDIO_PATH exists:", RAW_AUDIO_PATH.exists())
print("DIAGNOSIS_PATH exists:", DIAGNOSIS_PATH.exists())

RAW_AUDIO_PATH exists: True
DIAGNOSIS_PATH exists: True


In [6]:
features_df, mel_spec = load_and_preprocess_raw_audio_data()

print("features_df shape:", features_df.shape)
print("mel_spec shape:", mel_spec.shape)
print("diagnosis counts:")
print(features_df["diagnosis"].value_counts())
print("num patients:", features_df["patient_id"].nunique())


Skipped blacklisted patient 142

Skipped blacklisted patient 182

Skipped blacklisted patient 191

Skipped blacklisted patient 191

Skipped blacklisted patient 191
features_df shape: (3135, 44)
mel_spec shape: (3135, 128, 141, 1)
diagnosis counts:
diagnosis
COPD              2391
Pneumonia          238
Healthy            233
URTI               161
Bronchiectasis     112
Name: count, dtype: int64
num patients: 114


In [7]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(features_df["diagnosis"])

groups = features_df["patient_id"]
X = features_df.drop(columns=["patient_id", "diagnosis"], errors="ignore")

print("X shape:", X.shape)
print("y shape:", y.shape)
print("groups shape:", groups.shape)
print("Encoded classes:", list(label_encoder.classes_))
print("Feature columns:", X.columns.tolist()[:10], "...")

X shape: (3135, 42)
y shape: (3135,)
groups shape: (3135,)
Encoded classes: ['Bronchiectasis', 'COPD', 'Healthy', 'Pneumonia', 'URTI']
Feature columns: ['rms_mean', 'rms_std', 'zcr_mean', 'centroid_mean', 'centroid_std', 'flatness_mean', 'flatness_std', 'rolloff_mean', 'bandwidth_mean', 'flux_mean'] ...


### 🙋‍♀️ Preprocessing observations

The canonical preprocessing pipeline produces 3135 audio chunks from 114 patients across 5 classes. The resulting dataset is highly imbalanced, with COPD accounting for the large majority of examples. This makes macro F1 and class-wise metrics more informative than overall accuracy alone.

## 2. Split strategy

We split the data at the patient level, not the chunk level, to avoid data leakage (since each patient has multiple audio segments).

We use a two-stage split:

- **Outer split** → creates a held-out test set (never seen during training)
- **Inner split** → splits the remaining data into train and validation

Both splits use StratifiedGroupKFold so that:
- Patients don’t appear in multiple sets
- Class distribution is roughly preserved

Mental model = 
```
All patients
→ Train/Val + Test
→ Train + Validation
```

In [9]:
# Confirm group splits are correct
print("Unique patients:", groups.nunique())
print("Total samples:", len(groups))

Unique patients: 114
Total samples: 3135


In [11]:
# --- OUTER SPLIT: train+validation vs test ---
outer_sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
outer_train_val_idx, outer_test_idx = next(outer_sgkf.split(X, y, groups))

# train+validation subsets
X_train_val = X.iloc[outer_train_val_idx].reset_index(drop=True)
mel_spec_train_val = mel_spec[outer_train_val_idx]
y_train_val = y[outer_train_val_idx]
groups_train_val = groups.iloc[outer_train_val_idx].reset_index(drop=True)

# test subsets
X_test = X.iloc[outer_test_idx].reset_index(drop=True)
mel_spec_test = mel_spec[outer_test_idx]
y_test = y[outer_test_idx]
groups_test = groups.iloc[outer_test_idx].reset_index(drop=True)

# ---- INNER SPLIT: train vs validation ----
inner_sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
inner_train_idx, inner_val_idx = next(inner_sgkf.split(X_train_val, y_train_val, groups_train_val))

# train subsets
X_train = X_train_val.iloc[inner_train_idx].reset_index(drop=True)
mel_spec_train = mel_spec_train_val[inner_train_idx]
y_train = y_train_val[inner_train_idx]
groups_train = groups_train_val.iloc[inner_train_idx].reset_index(drop=True)

# validation subsets
X_val = X_train_val.iloc[inner_val_idx].reset_index(drop=True)
mel_spec_val = mel_spec_train_val[inner_val_idx]
y_val = y_train_val[inner_val_idx]
groups_val = groups_train_val.iloc[inner_val_idx].reset_index(drop=True)

print("Train chunks:", len(X_train))
print("Val chunks:", len(X_val))
print("Test chunks:", len(X_test))

print("Train patients:", groups_train.nunique())
print("Val patients:", groups_val.nunique())
print("Test patients:", groups_test.nunique())

Train chunks: 1407
Val chunks: 604
Test chunks: 1124
Train patients: 51
Val patients: 25
Test patients: 38


In [12]:
# Check for patient overlap between splits

train_patients = set(groups_train)
val_patients = set(groups_val)
test_patients = set(groups_test)

print("Train ∩ Val:", len(train_patients & val_patients))
print("Train ∩ Test:", len(train_patients & test_patients))
print("Val ∩ Test:", len(val_patients & test_patients))

Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


In [13]:
# Inspect class distribution per split

def show_split_distribution(name, y_split, label_encoder):
    counts = pd.Series(label_encoder.inverse_transform(y_split)).value_counts().sort_index()
    proportions = (counts / counts.sum()).round(3)
    df = pd.DataFrame({
        "count": counts,
        "proportion": proportions
    })
    print(f"\n{name}")
    print(df)

show_split_distribution("Train", y_train, label_encoder)
show_split_distribution("Validation", y_val, label_encoder)
show_split_distribution("Test", y_test, label_encoder)


Train
                count  proportion
Bronchiectasis     28       0.020
COPD             1166       0.829
Healthy           108       0.077
Pneumonia          28       0.020
URTI               77       0.055

Validation
                count  proportion
Bronchiectasis     14       0.023
COPD              486       0.805
Healthy            41       0.068
Pneumonia          21       0.035
URTI               42       0.070

Test
                count  proportion
Bronchiectasis     70       0.062
COPD              739       0.657
Healthy            84       0.075
Pneumonia         189       0.168
URTI               42       0.037


### 🙋‍♀️ Split observations

- No patient overlap between splits (no leakage)
- Train and validation sets have similar class distributions
- Test set is more balanced, with a significantly lower proportion of COPD and higher representation of minority classes (e.g. pneumonia ~ 17%)
- This makes test evaluation more challenging, but likely closer to real-world performance

## 3. Baseline

### Early logistic regression

As part of early data exploration, an initial logistic regression model was built using a wide set of features including:

- Demographic data (age, sex, BMI)
- Annotation features (crackles, wheezes, chest location)
- MFCC summary statistics

This was run on a 6-class version of the dataset.

**Key observations**:

- Accuracy ~0.78, but macro F1 ~0.3
- Strong performance on COPD, very weak performance on minority classes
- Numerical instability due to feature scaling and high dimensionality

**Conclusion**:

→ A simple linear model struggled to learn meaningful class separation, particularly under class imbalance, motivating the use of more expressive models.

### NEW Baseline model (logistic regression, aligned with final pipeline)

To establish a fair benchmark, a logistic regression model is trained using the same input features as the XGBoost model:

- 42 engineered audio features per chunk
- 5-class classification setup
- patient-level grouped splitting
- feature scaling applied

This allows us to isolate the impact of model complexity while keeping inputs constant.

In [14]:
baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000))
])

baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test)

print(classification_report(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))

              precision    recall  f1-score   support

           0       1.00      0.64      0.78        70
           1       0.74      0.98      0.85       739
           2       0.63      0.55      0.59        84
           3       0.88      0.08      0.15       189
           4       0.36      0.12      0.18        42

    accuracy                           0.74      1124
   macro avg       0.72      0.47      0.51      1124
weighted avg       0.76      0.74      0.68      1124

Macro F1: 0.5077545564772212


In [15]:
# ---- Evaluate at patient level ----

# Get probabilities for better aggregation
y_proba = baseline_model.predict_proba(X_test)

df_pred = pd.DataFrame({
    "patient_id": groups_test,
    "true": y_test
})

for i in range(y_proba.shape[1]):
    df_pred[f"proba_{i}"] = y_proba[:, i]

# Aggregate by patient (mean probas)
df_patient = df_pred.groupby("patient_id").mean()

# Final prediction
y_pred_patient = np.argmax(df_patient[[col for col in df_patient.columns if "proba_" in col]].values, axis=1)
y_true_patient = df_patient["true"].round().astype(int)

print(classification_report(y_true_patient, y_pred_patient))
print("Macro F1:", f1_score(y_true_patient, y_pred_patient, average="macro"))

              precision    recall  f1-score   support

           0       1.00      0.33      0.50         3
           1       0.63      1.00      0.77        17
           2       0.75      0.60      0.67        10
           3       0.00      0.00      0.00         3
           4       0.50      0.20      0.29         5

    accuracy                           0.66        38
   macro avg       0.58      0.43      0.45        38
weighted avg       0.62      0.66      0.60        38

Macro F1: 0.445021645021645


/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parame

### 💡 Baseline results and class imbalance

The dataset is highly imbalanced, with COPD accounting for ~88% samples after filtering to the final 5 classes. This has a strong impact on model behaviour and evaluation.

Baseline (logistic regression) results:

- Chunk-level: Accuracy ~0.74, Macro F1 ~0.51
- Patient-level: Accuracy ~0.66, Macro F1 ~0.45

Key observations:

- The model achieves very high recall for COPD, partly due to class imbalance and a tendency to default to the majority class when uncertain
- Performance on minority classes (e.g. Pneumonia, URTI) is significantly weaker, with some classes not predicted at all
- Macro F1 is substantially lower than accuracy, highlighting poor balance across classes
- Patient-level evaluation further reduces performance, giving a more realistic view of model behaviour

👉 Conclusion: **Even with well-engineered audio features, a simple linear model struggles to separate minority and clinically similar classes. More expressive models are needed, but improvements are expected to be incremental due to dataset limitations.**

## 4. XGBoost path

To reflect the project team’s implemented approach as closely as possible, XGBoost is trained using the same core hyperparameters as the application pipeline, including class-balanced sample weights on the training set and early stopping on the validation set.

In [19]:
num_classes = len(np.unique(y_train))
num_classes

5

In [20]:
# Build XGB Model
xgb_model = XGBClassifier(
    n_estimators=600,               # maximum number of trees
    max_depth=3,                    # maximum depth of each tree
    subsample=0.8,                  # row sampling to prevent overfitting
    colsample_bytree=0.7,           # feature sampling to prevent overfitting
    reg_lambda=1.5,                 # L2 regularisation (penalise complexity, helps generalisation)
    objective="multi:softprob",
    num_class=num_classes,
    random_state=42,
    eval_metric="mlogloss",         # use log loss for early stopping
    early_stopping_rounds=15
)

# Compute sample weights for imbalanced classes
w_train = compute_sample_weight(class_weight="balanced", y=y_train)

# Train XGB model
xgb_model.fit(
    X_train,
    y_train,
    sample_weight=w_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.7
,device,None
,early_stopping_rounds,15
,enable_categorical,False
,eval_metric,'mlogloss'


### 🙋‍♀️ Chunk vs patient-level evaluation

The model operates on short audio segments (chunks), but the task is to classify patients.

- Chunk-level evaluation measures performance on individual audio segments
- Patient-level evaluation aggregates predictions across all segments for a patient

Chunk-level metrics are typically higher but can be misleading due to noise and repeated samples from the same patient. Patient-level evaluation provides a more realistic measure of clinical performance.

In [21]:
# --- Chunk-level predictions and evaluation ---
y_pred_xgb = xgb_model.predict(X_test)

print("XGB Chunk-level Classification Report:")
print(classification_report(y_test, y_pred_xgb))
print("XGB Chunk-level Macro F1:", f1_score(y_test, y_pred_xgb, average="macro"))

XGB Chunk-level Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.23      0.37        70
           1       0.79      0.95      0.87       739
           2       0.39      0.69      0.50        84
           3       0.80      0.19      0.31       189
           4       0.04      0.02      0.03        42

    accuracy                           0.73      1124
   macro avg       0.59      0.42      0.41      1124
weighted avg       0.75      0.73      0.68      1124

XGB Chunk-level Macro F1: 0.41463560652201625


In [22]:
# --- Patient-level predictions and evaluation ---
y_proba_xgb = xgb_model.predict_proba(X_test)

# Build DataFrame and add probabilities per class
df_patient_xgb = pd.DataFrame({
  "patient_id": groups_test,
  "true": y_test
})

for i in range(y_proba_xgb.shape[1]):
    df_patient_xgb[f"prob_class_{i}"] = y_proba_xgb[:, i]

# Aggregate by patient (mean probas)
df_patient_xgb_agg = df_patient_xgb.groupby("patient_id").mean()

# Final prediction per patient
y_pred_patient_xgb = np.argmax(
    df_patient_xgb_agg[[col for col in df_patient_xgb_agg.columns if "prob_class_" in col]].values,
    axis=1
)

# True labels per patient
y_true_patient_xgb = df_patient_xgb_agg["true"].round().astype(int)

# Evaluate
print("XGB Patient-level Classification Report:")
print(classification_report(y_true_patient_xgb, y_pred_patient_xgb))
print("XGB Patient-level Macro F1:", f1_score(y_true_patient_xgb, y_pred_patient_xgb, average="macro"))

XGB Patient-level Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.33      0.50         3
           1       0.70      0.94      0.80        17
           2       0.64      0.70      0.67        10
           3       1.00      0.33      0.50         3
           4       0.00      0.00      0.00         5

    accuracy                           0.66        38
   macro avg       0.67      0.46      0.49        38
weighted avg       0.64      0.66      0.61        38

XGB Patient-level Macro F1: 0.49333333333333335


### 💡 XGBoost results

Chunk-level:
- Macro F1 decreased compared to the logistic baseline (~0.41 vs ~0.51)
- Performance remains dominated by COPD, with weak performance on minority classes

Patient-level:
- Slight improvement in macro F1 (~0.49 vs ~0.45)
- Indicates more consistent predictions across chunks

Key observations:
- XGBoost produces more stable predictions, improving aggregation at the patient level
- However, it does not meaningfully improve detection of minority classes
- URTI performance is particularly poor, with near-zero recall

👉 Conclusion: **Increasing model complexity does not significantly improve performance given the current dataset. Limitations are driven primarily by class imbalance and class separability rather than model choice.**

## 5. CNN path

The CNN is trained on mel spectrograms generated from the same 6-second audio chunks used elsewhere in the pipeline. This tests whether a deep learning model can learn useful time-frequency patterns directly from the audio representation, beyond the engineered tabular features used by the baseline and XGBoost models.

The CNN uses four convolutional blocks to learn increasingly abstract patterns from mel spectrograms, followed by global pooling and a small dense layer for final classification. Early stopping is used to limit overfitting on the validation set.

In [39]:
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights_dict = {cls: weight for cls, weight in zip(classes, class_weights)}
print(class_weights_dict)

{0: 10.05, 1: 0.24133790737564323, 2: 2.6055555555555556, 3: 10.05, 4: 3.6545454545454548}


In [40]:
# Number of output classes
num_classes = len(np.unique(y_train))

# One-hot encode labels for softmax classification
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)

# CNN input shape, e.g. (128, 141, 1)
input_shape = mel_spec_train.shape[1:]

# --- Build CNN model ---
cnn_model = models.Sequential()

# Input
cnn_model.add(layers.Input(shape=input_shape))

# Conv2D Block 1
cnn_model.add(layers.Conv2D(32, (3, 3), padding="same"))
cnn_model.add(layers.BatchNormalization())
cnn_model.add(layers.Activation("relu"))
cnn_model.add(layers.MaxPooling2D((2, 2)))

# Conv2D Block 2
cnn_model.add(layers.Conv2D(64, (3, 3), padding="same"))
cnn_model.add(layers.BatchNormalization())
cnn_model.add(layers.Activation("relu"))
cnn_model.add(layers.MaxPooling2D((2, 2)))

# Conv2D Block 3
cnn_model.add(layers.Conv2D(128, (3, 3), padding="same"))
cnn_model.add(layers.BatchNormalization())
cnn_model.add(layers.Activation("relu"))
cnn_model.add(layers.MaxPooling2D((2, 2)))

# Conv2D Block 4
cnn_model.add(layers.Conv2D(256, (3, 3), padding="same"))
cnn_model.add(layers.BatchNormalization())
cnn_model.add(layers.Activation("relu"))
cnn_model.add(layers.MaxPooling2D((2, 2)))

# Turn feature maps into one vector
cnn_model.add(layers.GlobalMaxPooling2D())

# Dense layer before classification
cnn_model.add(layers.Dense(32, activation="relu"))
cnn_model.add(layers.Dropout(0.3))

# Final prediction layer
cnn_model.add(layers.Dense(num_classes, activation="softmax"))

cnn_model.compile(
        optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
    )

# Early stopping callback
callbacks = [EarlyStopping(
    patience=5,
    restore_best_weights=True
)]

# Train CNN model
history = cnn_model.fit(
    mel_spec_train,
    y_train_cat,
    validation_data=(mel_spec_val, y_val_cat),
    epochs=30,
    batch_size=32,
    class_weight=class_weights_dict,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 7s 117ms/step - accuracy: 0.3333 - loss: 4.7572 - val_accuracy: 0.0679 - val_loss: 2.1047
Epoch 2/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.3269 - loss: 3.6888 - val_accuracy: 0.4487 - val_loss: 1.3215
Epoch 3/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 5s 107ms/step - accuracy: 0.3447 - loss: 3.1937 - val_accuracy: 0.0778 - val_loss: 3.7426
Epoch 4/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 5s 107ms/step - accuracy: 0.3085 - loss: 3.2940 - val_accuracy: 0.1093 - val_loss: 1.9576
Epoch 5/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 99ms/step - accuracy: 0.4009 - loss: 2.9648 - val_accuracy: 0.0629 - val_loss: 4.6743
Epoch 6/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 98ms/step - accuracy: 0.4684 - loss: 2.5484 - val_accuracy: 0.0248 - val_loss: 7.5149
Epoch 7/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.5117 - loss: 2.5455 - val_accuracy: 0.7666 - val_loss: 0.7355
Epoch 8/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.4890 - loss: 2.3295 - val_accuracy: 0.28

In [41]:
# Chunk level class predictions and evaluation
proba_test_cnn = cnn_model.predict(mel_spec_test, verbose=0)

# Convert probabilities to class predictions
pred_test_cnn = np.argmax(proba_test_cnn, axis=1)

print("CNN Chunk-level Classification Report:")
print(classification_report(y_test, pred_test_cnn))
print("CNN Chunk-level Macro F1:", f1_score(y_test, pred_test_cnn, average="macro"))

CNN Chunk-level Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        70
           1       0.75      0.89      0.82       739
           2       0.28      0.43      0.34        84
           3       0.96      0.28      0.43       189
           4       0.03      0.05      0.04        42

    accuracy                           0.67      1124
   macro avg       0.41      0.33      0.33      1124
weighted avg       0.68      0.67      0.64      1124

CNN Chunk-level Macro F1: 0.32636617395059375


In [42]:
# Patient level predictions and evaluation
df_patient_cnn = pd.DataFrame({
    "patient_id": groups_test.reset_index(drop=True),
    "true": y_test
})

for i in range(proba_test_cnn.shape[1]):
    df_patient_cnn[f"proba_class_{i}"] = proba_test_cnn[:, i]

# Aggregate by patient (mean probas)
df_patient_cnn_agg = df_patient_cnn.groupby("patient_id").mean()

# Final prediction per patient
y_pred_patient_cnn = np.argmax(
    df_patient_cnn_agg[[c for c in df_patient_cnn_agg.columns if c.startswith("proba_")]].values,
    axis=1
)

# True labels per patient
y_true_patient_cnn = df_patient_cnn_agg["true"].round().astype(int)

# Evaluate
print("CNN Patient-level Classification Report:")
print(classification_report(y_true_patient_cnn, y_pred_patient_cnn))
print("CNN Patient-level Macro F1:", f1_score(y_true_patient_cnn, y_pred_patient_cnn, average="macro"))

CNN Patient-level Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.58      0.88      0.70        17
           2       0.44      0.40      0.42        10
           3       1.00      0.33      0.50         3
           4       0.00      0.00      0.00         5

    accuracy                           0.53        38
   macro avg       0.40      0.32      0.32        38
weighted avg       0.45      0.53      0.46        38

CNN Patient-level Macro F1: 0.3237454100367197


/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parame

### 💡 CNN results

A class-weighted CNN produced more stable behaviour than earlier unweighted CNN runs explored in development notebooks, avoiding complete collapse to the majority class.

- Chunk-level: Accuracy ~0.67, Macro F1 ~0.33
- Patient-level: Accuracy ~0.53, Macro F1 ~0.32

Key observations:
- Class weighting improved robustness and allowed the model to recover some minority-class signal
- COPD remained the strongest class
- Healthy and Pneumonia showed modest signal
- Bronchiectasis and URTI remained very weak

👉 Conclusion: **The CNN remains substantially weaker than the tabular models as a standalone classifier, but may still contribute complementary signal in the hybrid model.**

## 6. Hybrid path

run weighted probability fusion

confirm weight used

compare against XGB-only

In [43]:
# Combine XGB and CNN predictions (late fusion)
w=DEFAULT_XGB_WEIGHT

hybrid_proba_test = w * y_proba_xgb + (1 - w) * proba_test_cnn

# Convert probabilities to class predictions
hybrid_pred_test = np.argmax(hybrid_proba_test, axis=1)

In [44]:
# Chunk level predictions and evaluation
print("Hybrid Chunk-level Classification Report:")
print(classification_report(y_test, hybrid_pred_test))
print("Hybrid Chunk-level Macro F1:", f1_score(y_test, hybrid_pred_test, average="macro"))

Hybrid Chunk-level Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.17      0.29        70
           1       0.78      0.96      0.86       739
           2       0.43      0.68      0.52        84
           3       0.93      0.21      0.34       189
           4       0.05      0.02      0.03        42

    accuracy                           0.73      1124
   macro avg       0.61      0.41      0.41      1124
weighted avg       0.76      0.73      0.68      1124

Hybrid Chunk-level Macro F1: 0.4077717174647934


In [ ]:
# Patient-level predictions and evaluation
df_patient_hybrid = pd.DataFrame({
    "patient_id": groups_test.reset_index(drop=True),
    "true": y_test
})

for i in range(hybrid_proba_test.shape[1]):
    df_patient_hybrid[f"proba_class_{i}"] = hybrid_proba_test[:, i]

# Aggregate by patient (mean probas)
df_patient_hybrid_agg = df_patient_hybrid.groupby("patient_id").mean()

# Final prediction per patient
y_pred_patient_hybrid = np.argmax(
    df_patient_hybrid_agg[[c for c in df_patient_hybrid_agg.columns if c.startswith("proba_")]].values,
    axis=1
)

# True labels per patient
y_true_patient_hybrid = df_patient_hybrid_agg["true"].round().astype(int)

print("Hybrid Patient-level Classification Report:")
print(classification_report(y_true_patient_hybrid, y_pred_patient_hybrid))
print("Hybrid Patient-level Macro F1:", f1_score(y_true_patient_hybrid, y_pred_patient_hybrid, average="macro"))

Hybrid Patient-level Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.64      0.94      0.76        17
           2       0.70      0.70      0.70        10
           3       1.00      0.33      0.50         3
           4       0.00      0.00      0.00         5

    accuracy                           0.63        38
   macro avg       0.47      0.39      0.39        38
weighted avg       0.55      0.63      0.56        38

Hybrid Patient-level Macro F1: 0.3923809523809524


/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parame

### 💡 Hybrid model (late fusion) results

Chunk-level:
- Macro F1 ~0.41 (no improvement over XGBoost)

Patient-level:
- Macro F1 ~0.39 (worse than XGBoost ~0.49)

Key observations:
- Combining CNN and XGBoost predictions did not improve performance
- The CNN signal was too weak and inconsistent to enhance the stronger tabular model
- Hybrid predictions slightly degraded performance relative to XGBoost alone

👉 Conclusion: **In this setting, late fusion does not provide additional value. Model performance is primarily driven by the tabular features, and limitations appear to stem from dataset imbalance and class separability rather than model architecture.**

## 7. Re-evaluating model selection (post project)

Final model selection: **XGBoost** 🏆

⚠️ This model selection reflects the results of the current re-evaluation, not the original project implementation.

The original project used a hybrid (late fusion) approach combining CNN and XGBoost models. The selection below reflects a separate re-evaluation of the pipeline using a stricter and more consistent methodology.

Based on chunk-level and patient-level metrics, XGBoost emerged as the strongest standalone model. It was therefore retrained on the full dataset for final inference in this analysis.

Reconstructing the hybrid approach under the same evaluation conditions showed no performance improvement over XGBoost.

Model        | Chunk F1 | Patient F1
-------------|----------|------------
Logistic     | 0.51     | 0.45
XGBoost      | 0.41     | 0.49
CNN          | 0.33     | 0.32
Hybrid       | 0.41     | 0.39

## 8. Run Summary

**Example predictions (single-file inference)**

As a qualitative check, the final model was run on three example recordings used during development, covering COPD, Healthy, and Pneumonia cases.

These examples illustrate how the model behaves on individual inputs, but are not used as a formal evaluation metric.


*e.g. Final rerun used X classes, 6-second chunks, grouped patient-level splitting, XGB + CNN weighted probability fusion, and produced chunk-level accuracy of ~A / macro F1 ~B, with patient-level accuracy of ~C / macro F1 ~D.*

concise final block with:

classes used

split strategy

chunk length

final metric snapshot

main caveats

In [ ]:
# Save XGBoost model and metadata
from pathlib import Path
import joblib
import json

repo_root = Path().resolve().parent
models_path = repo_root / "models"

# ensure folder exists
models_path.mkdir(exist_ok=True)

# save model
joblib.dump(xgb_model, models_path / "xgb_final.pkl")

# save feature columns
joblib.dump(X_train.columns.tolist(), models_path / "feature_columns.pkl")

# save class names
with open(models_path / "class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f)

print("Clean model package saved ✅")

Clean model package saved ✅


In [ ]:
# Load model and metadata
loaded_xgb_model = joblib.load(models_path / "xgb_final.pkl")
loaded_xgb_columns = joblib.load(models_path / "feature_columns.pkl")

with open(models_path / "class_names.json", "r") as f:
    loaded_class_names = json.load(f)

print("All assets load ✅")
print(loaded_class_names)

All assets load ✅
['Bronchiectasis', 'COPD', 'Healthy', 'Pneumonia', 'URTI']


In [50]:
# ==============================
# LOAD AND PREDICT DEMO DATA
# ==============================
from pathlib import Path
import json
import joblib
import numpy as np
import librosa as lb
import pandas as pd

from smart_stethoscope.params import TARGET_SAMPLING_RATE, repo_root
from smart_stethoscope.interface.main import preprocess_for_prediction

# --- Load three demo audio files (not used in training) and preprocess ---
copd_path = "/Users/keira/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/142_1b1_Pl_mc_LittC2SE.wav"
healthy_path = "/Users/keira/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/182_1b1_Tc_sc_Meditron.wav"
pneumonia_path = "/Users/keira/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/191_2b1_Pl_mc_LittC2SE.wav"

demo_examples = [
    {
        "name": "COPD example",
        "true_label": "COPD",
        "audio_path": copd_path,
        "start": 0.036,
        "end": 19.964
    },
    {
        "name": "Healthy example",
        "true_label": "Healthy",
        "audio_path": healthy_path,
        "start": 0.079,
        "end": 19.997
    },
    {
        "name": "Pneumonia example",
        "true_label": "Pneumonia",
        "audio_path": pneumonia_path,
        "start": 0.05,
        "end": 19.979
    }
]

# --- Helper function to load and preprocess a single audio file for prediction ---
def predict_single_file_xgb(audio_path: Path, start: float, end: float):

    # Load raw audio
    audio, sr = lb.load(audio_path, sr=TARGET_SAMPLING_RATE)

    # Preprocess into chunk-level tabular features + mel specs (ignore mel specs)
    xgb_features, _ = preprocess_for_prediction(audio, sr, start, end)

    if xgb_features.empty:
        raise ValueError(f"No valid segments produced for {audio_path.name}")

    # Align columns to training-time order
    xgb_features = xgb_features.reindex(columns=loaded_xgb_columns)

    # Predict chunk probabilities
    chunk_proba = xgb_model.predict_proba(xgb_features)

    # Aggregate across chunks
    mean_proba = chunk_proba.mean(axis=0)

    # Final prediction
    pred_idx = int(np.argmax(mean_proba))
    pred_label = loaded_class_names[pred_idx]
    confidence = float(mean_proba[pred_idx])

    return {
        "n_chunks": len(xgb_features),
        "predicted_label": pred_label,
        "confidence": confidence,
        "mean_proba": mean_proba,
    }

# --- Loop through demo examples, predict with XGB, and print results ---
for demo in demo_examples:
    result = predict_single_file_xgb(
        audio_path=demo["audio_path"],
        start=demo["start"],
        end=demo["end"],
    )

    print("=" * 60)
    print(demo["name"])
    print("True label: ", demo["true_label"])
    print("Predicted:  ", result["predicted_label"])
    print("Confidence: ", round(result["confidence"], 3))
    print("Chunks:     ", result["n_chunks"])
    print("\nClass probabilities:")

    for label, proba in zip(loaded_class_names, result["mean_proba"]):
        print(f"  {label:<15} {proba:.3f}")

    print()

COPD example
True label:  COPD
Predicted:   COPD
Confidence:  0.324
Chunks:      3

Class probabilities:
  Bronchiectasis  0.196
  COPD            0.324
  Healthy         0.221
  Pneumonia       0.082
  URTI            0.177

Healthy example
True label:  Healthy
Predicted:   Healthy
Confidence:  0.424
Chunks:      3

Class probabilities:
  Bronchiectasis  0.040
  COPD            0.191
  Healthy         0.424
  Pneumonia       0.088
  URTI            0.257

Pneumonia example
True label:  Pneumonia
Predicted:   Healthy
Confidence:  0.519
Chunks:      3

Class probabilities:
  Bronchiectasis  0.030
  COPD            0.113
  Healthy         0.519
  Pneumonia       0.156
  URTI            0.182



### 🧪 Example file inference check

As a qualitative smoke test, the retrained XGBoost model was run on three example recordings representing COPD, Healthy, and Pneumonia cases.

Results:
- COPD example → predicted COPD (low confidence)
- Healthy example → predicted Healthy
- Pneumonia example → misclassified as Healthy

This aligns with the broader evaluation results:
- the model can identify some clearer classes
- confidence is often diffuse across multiple diagnoses
- Pneumonia remains difficult to separate reliably

## Conclusion

This rerun showed that the original project’s core challenge was not a lack of model sophistication, but the structure of the dataset itself.

Using a consistent grouped split and comparing models on both chunk-level and patient-level metrics, the strongest standalone result came from XGBoost. The late-fusion hybrid model used in the original project was reconstructed faithfully, but did not outperform XGBoost in this evaluation.

Across all models, performance plateaued relatively early:
- The logistic baseline already captured much of the available signal  
- XGBoost provided the best balance of stability and performance, particularly at the patient level  
- The CNN struggled as a standalone model and was highly sensitive to class imbalance  
- The hybrid (late fusion) approach did not improve performance over XGBoost  

These results suggest that model complexity was not the primary constraint. Instead, performance was limited by:
- **Severe class imbalance**
- **Overlapping class characteristics**
- **Limited data for minority classes**

A qualitative inference check on example recordings showed that the model can identify clearer cases, but confidence is often diffuse and more ambiguous conditions (e.g. Pneumonia) remain difficult to distinguish reliably.

**Key findings**:

- The dataset remains heavily imbalanced even after reducing the class set
- Simpler models already capture much of the available signal
- The CNN is unstable as a standalone model and highly sensitive to imbalance handling
- Late fusion does not reliably add enough complementary signal to justify the added complexity

The re-evaluation also demonstrates that simpler models can be competitive, and that additional complexity (e.g. hybrid models) does not guarantee improved outcomes.

*This does not invalidate the original project. Rather, it shows that the original hybrid choice was a reasonable hypothesis under project constraints, but that stricter re-evaluation leads to a different model selection conclusion today.*
